[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/14.%20The%20Vessel%20Stowage%20Planning%20Problem%20%28Export%29/P14-Tier-7_executed.ipynb)

---


# The Vessel Stowage Planning Problem (Export) — Tier 7
## Human–AI Symbiotic Partnership

### Purpose and learning objectives

This notebook develops a transparent decision-support workflow for a small vessel-stowage example. An explainable AI heuristic scores candidate slots, a human-expert system reviews the recommendation, and an authority-allocation rule records how the decision is governed. The example then tracks trust-related indicators and summarizes the simulated decisions.

By the end, you should be able to follow how the candidate-slot score is assembled, how expert opinions are aggregated, how authority is allocated, and how the resulting records are analyzed.

### Inputs and outputs

The inputs are a vessel state, a sequence of containers, candidate slot identifiers, and a small team of expert profiles. The outputs are AI recommendations, expert opinions, decision records, heuristic performance summaries, and diagnostic plots. The notebook uses ordinary Python objects and does not require external data files.

### Scope and modeling assumptions

This is an illustrative decision-support simulation, not a complete stowage optimizer or a validated operational safety system. The AI score is a weighted heuristic; it does not enforce vessel stability, hazardous-material segregation, reefer power, container compatibility, or structural limits as hard constraints. The scenario's slot identifiers encode the bay in their first two characters. Candidate slots are treated as distinct positions, and a selected position is removed before the next container is considered.

Human opinions, authority allocation, trust changes, and estimated benefits are rule-based proxies, not measurements of real planner performance or empirically calibrated quantities. A human recommendation may call for modification, but this example does not model a human-selected replacement slot: the recorded recommendation remains the AI-suggested slot. The simulated benefit is an illustrative estimate, not a realized financial saving. The model does not establish claims about actual vessel capacity, safety, or comparative performance.

### Method at a glance

The workflow keeps four stages distinct: (1) score each available slot and select the highest-scoring one; (2) obtain opinions from relevant experts and compute their mean agreement score; (3) apply fixed rules to assign decision authority; and (4) record the decision and update heuristic trust indicators. This is a rule-based human–AI collaboration model, not an exact mathematical optimization method.

For a container \(c\), candidate slot \(s\), and vessel state \(v\), the AI score is

$$
S(c,s,v) = w_o O(c,v) + w_s B(c,s,v) + w_e E(s,v) + w_h H(c,s),
$$

where \(O\) is the overstowage-related component, \(B\) is a simplified bay-weight balance component, \(E\) is a loading-distance efficiency component, and \(H\) is the hazardous-container safety component. The weights are \(w_o=0.4\), \(w_s=0.3\), \(w_e=0.2\), and \(w_h=0.1\). The safety component is zero for non-hazardous containers. The terms are heuristic scores, not probabilities or guaranteed physical outcomes.

If \(A\) is the set of available slots, the recommendation is

$$
s^* = \operatorname*{arg\,max}_{s\in A} S(c,s,v).
$$

The implementation evaluates every slot in `available_slots`, sorts the resulting scores in descending order, and chooses the first. Tied scores are resolved by the order in which slots occur in the input list. Its scoring work is \(O(|A|)\), apart from sorting the scores, which takes \(O(|A|\log|A|)\).

In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple
import random

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")

In [ ]:
@dataclass
class HumanExpert:
    """Profile of a human expert participating in the review process."""

    id: str
    name: str
    experience_years: int
    specialization: str
    confidence_level: float
    vessel_knowledge: Dict[str, float]
    risk_tolerance: float
    decision_speed: float
    accuracy_history: List[float] = field(default_factory=list)


@dataclass
class AIRecommendation:
    """AI slot recommendation together with its heuristic evidence."""

    recommendation_id: str
    action_type: str
    container_id: str
    suggested_slot: str
    confidence_score: float
    uncertainty_range: Tuple[float, float]
    reasoning: List[str]
    alternatives: List[Dict[str, Any]]
    risk_assessment: Dict[str, float]
    expected_benefit: float
    recommendation_score: float


@dataclass
class TrustMetrics:
    """Heuristic indicators recorded during collaboration."""

    human_trust_in_ai: float = 0.5
    ai_confidence_in_human: float = 0.5
    collaboration_history: List[Dict[str, Any]] = field(default_factory=list)
    disagreement_count: int = 0
    agreement_count: int = 0
    successful_collaborations: int = 0


@dataclass
class DecisionRecord:
    """Record of one AI recommendation and its human review."""

    timestamp: datetime
    problem_context: str
    ai_recommendation: AIRecommendation
    human_input: Dict[str, Any]
    final_decision: str
    decision_authority: str
    outcome_confidence: float
    actual_outcome: Optional[float] = None
    lessons_learned: List[str] = field(default_factory=list)

## 1. Explainable AI slot scoring

The scoring rule ranks the currently available positions; it does not prove that the highest-ranked position is feasible in a real vessel. Separating the score components makes the recommendation inspectable: each component can be traced to its inputs, and the reasoning text can be reviewed alongside the numeric ranking. The four weights express the illustrative trade-offs used in this example.

In [ ]:
class ExplainableAIEngine:
    """Rank candidate slots and provide heuristic explanations."""

    def __init__(self, model_confidence: float = 0.85):
        self.model_confidence = model_confidence
        self.explanation_templates = self._initialize_explanations()
        self.constraint_weights = {
            "overstowage": 0.4,
            "stability": 0.3,
            "efficiency": 0.2,
            "safety": 0.1,
        }

    def _initialize_explanations(self) -> Dict[str, List[str]]:
        """Store explanation templates associated with the score dimensions."""
        return {
            "overstowage_minimization": [
                "The destination's order in the discharge sequence contributes to the score."
            ],
            "stability_optimization": [
                "The simplified score compares the bay's resulting weight with an average-weight target."
            ],
            "hazardous_segregation": [
                "Hazardous-container safety is represented by a fixed bay-based score, not a segregation constraint."
            ],
            "efficiency_optimization": [
                "The distance from the current loading bay contributes to the score."
            ],
        }

    @staticmethod
    def _parse_bay(slot: str) -> int:
        """Read the two-character bay code at the start of a slot identifier."""
        if len(slot) < 2 or not slot[:2].isdigit():
            raise ValueError(f"Slot identifier must begin with a two-digit bay code: {slot!r}")
        return int(slot[:2])

    def generate_recommendation(
        self,
        container: Dict[str, Any],
        vessel_state: Dict[str, Any],
        available_slots: List[str],
    ) -> AIRecommendation:
        """Score candidate positions and return the highest-ranked one."""
        if not available_slots:
            raise ValueError("At least one available slot is required.")

        slot_scores = []
        for slot in available_slots:
            score = self._evaluate_slot(container, slot, vessel_state)
            slot_scores.append({"slot": slot, "score": score})

        # Stable sorting preserves the input order when two heuristic scores tie.
        slot_scores.sort(key=lambda item: item["score"], reverse=True)
        best_slot = slot_scores[0]["slot"]
        best_score = slot_scores[0]["score"]

        if len(slot_scores) > 1:
            score_gap = best_score - slot_scores[1]["score"]
            confidence = min(0.95, max(0.0, 0.5 + score_gap * 2))
        else:
            confidence = 0.7

        reasoning = self._generate_reasoning(container, best_slot, vessel_state)
        risk_assessment = self._assess_risks(container, best_slot, vessel_state)
        alternatives = slot_scores[1:4]
        expected_benefit = self._calculate_benefit(container, best_slot, vessel_state)

        return AIRecommendation(
            recommendation_id=f"AI_REC_{datetime.now().strftime('%H%M%S%f')}",
            action_type="place_container",
            container_id=container["id"],
            suggested_slot=best_slot,
            confidence_score=confidence,
            uncertainty_range=(best_score * 0.8, best_score * 1.2),
            reasoning=reasoning,
            alternatives=alternatives,
            risk_assessment=risk_assessment,
            expected_benefit=expected_benefit,
            recommendation_score=best_score,
        )

    def _evaluate_slot(
        self,
        container: Dict[str, Any],
        slot: str,
        vessel_state: Dict[str, Any],
    ) -> float:
        """Compute the weighted heuristic score for one candidate slot."""
        score = 0.0

        discharge_order = vessel_state["discharge_ports"].index(container["destination"])
        overstowage_penalty = discharge_order * 10
        score += (100 - overstowage_penalty) * self.constraint_weights["overstowage"]

        bay = self._parse_bay(slot)
        current_weight = vessel_state["weight_by_bay"].get(bay, 0.0)
        new_weight = current_weight + container["weight"]
        bay_count = len(vessel_state["weight_by_bay"])
        if bay_count == 0:
            raise ValueError("The vessel state must contain at least one bay weight.")
        ideal_weight = vessel_state["total_weight"] / bay_count
        weight_balance_score = max(0.0, 100 - abs(new_weight - ideal_weight))
        score += weight_balance_score * self.constraint_weights["stability"]

        distance = abs(bay - vessel_state["current_loading_bay"])
        efficiency_score = max(0.0, 100 - distance * 5)
        score += efficiency_score * self.constraint_weights["efficiency"]

        if container["type"] == "hazardous":
            safety_score = 90 if bay in [1, 22] else 70
            score += safety_score * self.constraint_weights["safety"]

        return score

    def _generate_reasoning(
        self,
        container: Dict[str, Any],
        slot: str,
        vessel_state: Dict[str, Any],
    ) -> List[str]:
        """Describe the main inputs that contribute to the chosen slot's score."""
        bay = self._parse_bay(slot)
        discharge_order = vessel_state["discharge_ports"].index(container["destination"])
        current_weight = vessel_state["weight_by_bay"].get(bay, 0.0)
        new_weight = current_weight + container["weight"]
        distance = abs(bay - vessel_state["current_loading_bay"])

        reasoning = [
            f"Container {container['id']} is destined for {container['destination']} "
            f"(port {discharge_order + 1} in the discharge sequence).",
            f"Slot {slot} is in Bay {bay}; its score includes the simplified discharge-order term.",
            f"Bay {bay} has {current_weight:.1f} tons before placement and "
            f"{new_weight:.1f} tons after adding this container.",
            f"The candidate bay is {distance} bay positions from the current loading bay.",
        ]

        if container["type"] == "hazardous":
            reasoning.append(
                "A fixed bay-based hazardous-container safety score is included; "
                "actual segregation requirements are not checked."
            )
        elif container["type"] == "reefer":
            reasoning.append(
                "Reefer power connections are outside this heuristic and require separate verification."
            )
        elif container["size"] == "45ft":
            reasoning.append(
                "The score does not validate 45-foot slot clearance; that requires separate verification."
            )

        return reasoning

    def _assess_risks(
        self,
        container: Dict[str, Any],
        slot: str,
        vessel_state: Dict[str, Any],
    ) -> Dict[str, float]:
        """Return illustrative risk indicators used by the expert-review rules."""
        risks = {
            "stability_risk": 0.1,
            "overstowage_risk": 0.05,
            "operational_risk": 0.1,
            "safety_risk": 0.02,
        }

        if container["type"] == "hazardous":
            risks["safety_risk"] = 0.15
        if container["weight"] > 25:
            risks["stability_risk"] = 0.2
        if self._parse_bay(slot) in [1, 22]:
            risks["stability_risk"] += 0.1

        return risks

    def _calculate_benefit(
        self,
        container: Dict[str, Any],
        slot: str,
        vessel_state: Dict[str, Any],
    ) -> float:
        """Calculate the scenario's illustrative benefit estimate."""
        discharge_order = vessel_state["discharge_ports"].index(container["destination"])
        overstowage_savings = (
            len(vessel_state["discharge_ports"]) - discharge_order - 1
        ) * 500

        bay = self._parse_bay(slot)
        distance = abs(bay - vessel_state["current_loading_bay"])
        time_savings = distance * 15
        efficiency_savings = time_savings * 2
        stability_allowance = 100

        return overstowage_savings + efficiency_savings + stability_allowance

## 2. Human expert review

An expert's specialization determines which review rules apply. The system selects up to three experts with the highest specialization scores, generates a rule-based opinion for each, and averages their agreement scores. The mean is a simple consensus proxy: it is not a statistical estimate of expert reliability. The optional senior-expert intuition rule uses Python's random generator, so that element of the simulation can vary between runs.

In [ ]:
class HumanExpertSystem:
    """Select relevant experts and aggregate their rule-based opinions."""

    def __init__(self, experts: List[HumanExpert]):
        if not experts:
            raise ValueError("At least one human expert is required.")
        self.experts = {expert.id: expert for expert in experts}
        self.expertise_matrix = self._build_expertise_matrix()

    def _build_expertise_matrix(self) -> Dict[str, Dict[str, float]]:
        """Assign illustrative relevance scores for each problem category."""
        return {
            "stability": {
                expert_id: 0.9 if expert.specialization == "stability" else 0.6
                for expert_id, expert in self.experts.items()
            },
            "hazardous": {
                expert_id: 0.9 if expert.specialization == "hazardous" else 0.5
                for expert_id, expert in self.experts.items()
            },
            "oversized": {
                expert_id: 0.9 if expert.specialization == "oversized" else 0.4
                for expert_id, expert in self.experts.items()
            },
            "operations": {
                expert_id: 0.9 if expert.specialization == "operations" else 0.7
                for expert_id, expert in self.experts.items()
            },
            "general": {
                expert_id: expert.confidence_level
                for expert_id, expert in self.experts.items()
            },
        }

    def get_expert_input(
        self,
        problem_type: str,
        ai_recommendation: AIRecommendation,
        context: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Collect selected expert opinions and summarize their review."""
        expertise_scores = self.expertise_matrix.get(
            problem_type, self.expertise_matrix["general"]
        )
        selected_ids = sorted(
            expertise_scores,
            key=lambda expert_id: expertise_scores[expert_id],
            reverse=True,
        )[:3]
        relevant_experts = [self.experts[expert_id] for expert_id in selected_ids]

        expert_opinions = [
            self._generate_expert_opinion(expert, ai_recommendation, context)
            for expert in relevant_experts
        ]
        consensus_score = float(
            np.mean([opinion["agreement_score"] for opinion in expert_opinions])
        )
        concerns = sorted(
            {concern for opinion in expert_opinions for concern in opinion["concerns"]}
        )
        suggestions = sorted(
            {
                suggestion
                for opinion in expert_opinions
                for suggestion in opinion["suggestions"]
            }
        )

        return {
            "expert_opinions": expert_opinions,
            "consensus_score": consensus_score,
            "concerns": concerns,
            "suggestions": suggestions,
            "recommendation": self._formulate_human_recommendation(
                consensus_score, concerns
            ),
        }

    def _generate_expert_opinion(
        self,
        expert: HumanExpert,
        ai_recommendation: AIRecommendation,
        context: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Apply specialization-specific review rules to one recommendation."""
        agreement_score = expert.confidence_level
        concerns = []
        suggestions = []

        if expert.specialization == "stability":
            if ai_recommendation.risk_assessment["stability_risk"] > 0.15:
                agreement_score -= 0.2
                concerns.append("High stability risk indicator")
                suggestions.append("Review an alternative slot and the vessel's weight limits")

        elif expert.specialization == "hazardous":
            if context.get("container_type") == "hazardous":
                if ai_recommendation.risk_assessment["safety_risk"] > 0.1:
                    agreement_score -= 0.3
                    concerns.append("Hazardous-material segregation needs separate verification")
                    suggestions.append("Verify the applicable hazardous-material segregation rules")

        elif expert.specialization == "operations":
            if ai_recommendation.expected_benefit < 200:
                agreement_score -= 0.1
                concerns.append("Low illustrative operational benefit estimate")
                suggestions.append("Review whether a closer loading bay is available")

        if expert.experience_years > 15 and random.random() < 0.3:
            suggestions.append("Check vessel-specific operational constraints")
            agreement_score += 0.1

        return {
            "expert_id": expert.id,
            "expert_name": expert.name,
            "specialization": expert.specialization,
            "agreement_score": max(0.0, min(1.0, agreement_score)),
            "concerns": concerns,
            "suggestions": suggestions,
            "reasoning": (
                f"Rule-based review informed by {expert.experience_years} years of "
                f"{expert.specialization} experience."
            ),
        }

    @staticmethod
    def _formulate_human_recommendation(
        consensus_score: float,
        concerns: List[str],
    ) -> str:
        """Translate the consensus proxy into a review label."""
        if consensus_score > 0.8 and not concerns:
            return "ACCEPT — no concerns were recorded by the selected experts"
        if consensus_score > 0.6:
            return "REVIEW — consider the recorded concerns and suggestions"
        if consensus_score > 0.4:
            return "COLLABORATE — resolve concerns before operational use"
        return "REJECT — seek an alternative after expert review"

## 3. Authority allocation and decision records

The authority rule combines the AI confidence score, human consensus proxy, problem category, and presence of expert concerns. It determines who governs the review process; it does not independently verify that a slot is safe or feasible. The outcome-confidence value is another heuristic, formed from a weighted average and reduced when concerns are present. Because these quantities are rule-based, they should be interpreted as indicators within this demonstration, not calibrated probabilities.

In [ ]:
class HumanAISymbioticPartnership:
    """Coordinate AI recommendations, human review, and decision records."""

    def __init__(self, experts: List[HumanExpert]):
        self.ai_engine = ExplainableAIEngine()
        self.human_system = HumanExpertSystem(experts)
        self.trust_metrics = TrustMetrics()
        self.decision_history: List[DecisionRecord] = []
        self.authority_allocation = {
            "high_confidence_ai": 0.8,
            "human_expertise": 0.7,
            "collaboration_zone": 0.5,
        }

    def make_collaborative_decision(
        self,
        container: Dict[str, Any],
        vessel_state: Dict[str, Any],
        available_slots: List[str],
    ) -> DecisionRecord:
        """Generate a recommendation, collect review, and record authority."""
        ai_rec = self.ai_engine.generate_recommendation(
            container, vessel_state, available_slots
        )
        problem_type = self._classify_problem(container, vessel_state)
        human_input = self.human_system.get_expert_input(
            problem_type,
            ai_rec,
            {
                "container_type": container["type"],
                "container_weight": container["weight"],
                "destination": container["destination"],
            },
        )
        decision_authority, final_decision = self._allocate_authority(
            ai_rec, human_input, problem_type
        )

        decision = DecisionRecord(
            timestamp=datetime.now(),
            problem_context=(
                f"Place {container['id']} ({container['type']}) "
                f"for {container['destination']}"
            ),
            ai_recommendation=ai_rec,
            human_input=human_input,
            final_decision=final_decision,
            decision_authority=decision_authority,
            outcome_confidence=self._calculate_outcome_confidence(ai_rec, human_input),
        )
        self._update_trust_metrics(decision)
        self.decision_history.append(decision)
        return decision

    @staticmethod
    def _classify_problem(container: Dict[str, Any], vessel_state: Dict[str, Any]) -> str:
        """Select a review category using the scenario's container attributes."""
        if container["type"] == "hazardous":
            return "hazardous"
        if container["size"] == "45ft":
            return "oversized"
        if container["weight"] > 25:
            return "stability"
        return "operations"

    def _allocate_authority(
        self,
        ai_rec: AIRecommendation,
        human_input: Dict[str, Any],
        problem_type: str,
    ) -> Tuple[str, str]:
        """Apply the predefined authority-allocation rules in order."""
        ai_confidence = ai_rec.confidence_score
        human_consensus = human_input["consensus_score"]

        if (
            ai_confidence > self.authority_allocation["high_confidence_ai"]
            and problem_type == "operations"
            and not human_input["concerns"]
        ):
            return "ai", "ai_led"
        if (
            problem_type in ["hazardous", "oversized"]
            and human_consensus > self.authority_allocation["human_expertise"]
        ):
            return "human", "human_led"
        if (
            abs(ai_confidence - human_consensus)
            < self.authority_allocation["collaboration_zone"]
            or human_input["concerns"]
        ):
            return "shared", "collaborative"
        return "ai", "ai_led_with_oversight"

    def _calculate_outcome_confidence(
        self,
        ai_rec: AIRecommendation,
        human_input: Dict[str, Any],
    ) -> float:
        """Combine AI confidence and human consensus using the stated heuristic."""
        ai_weight = 0.6 if self.trust_metrics.human_trust_in_ai > 0.7 else 0.4
        human_weight = 1 - ai_weight
        confidence = (
            ai_weight * ai_rec.confidence_score
            + human_weight * human_input["consensus_score"]
        )
        if human_input["concerns"]:
            confidence *= max(0.0, 1 - len(human_input["concerns"]) * 0.1)
        return max(0.0, min(1.0, confidence))

    def _update_trust_metrics(self, decision: DecisionRecord) -> None:
        """Update proxy trust indicators and preserve a history for plotting."""
        human_input = decision.human_input
        ai_confidence = decision.ai_recommendation.confidence_score
        human_consensus = human_input["consensus_score"]
        has_concerns = bool(human_input["concerns"])

        # In this demonstration, an opinion set without recorded concerns counts as agreement.
        if has_concerns:
            self.trust_metrics.disagreement_count += 1
        else:
            self.trust_metrics.agreement_count += 1
            self.trust_metrics.successful_collaborations += 1

        if ai_confidence > human_consensus and not has_concerns:
            self.trust_metrics.human_trust_in_ai += 0.05
        elif ai_confidence > human_consensus and has_concerns:
            self.trust_metrics.human_trust_in_ai -= 0.02

        self.trust_metrics.human_trust_in_ai = max(
            0.1, min(0.95, self.trust_metrics.human_trust_in_ai)
        )
        self.trust_metrics.collaboration_history.append(
            {
                "timestamp": decision.timestamp,
                "decision_authority": decision.decision_authority,
                "ai_confidence": ai_confidence,
                "human_consensus": human_consensus,
                "outcome_confidence": decision.outcome_confidence,
                "human_trust_in_ai": self.trust_metrics.human_trust_in_ai,
                "agreement": not has_concerns,
            }
        )

    def analyze_partnership_performance(self) -> Dict[str, Any]:
        """Summarize decision records and the proxy trust indicators."""
        if not self.decision_history:
            return {"status": "No decisions have been recorded."}

        authority_counts = defaultdict(int)
        for decision in self.decision_history:
            authority_counts[decision.decision_authority] += 1

        avg_ai_confidence = float(
            np.mean(
                [d.ai_recommendation.confidence_score for d in self.decision_history]
            )
        )
        avg_human_consensus = float(
            np.mean([d.human_input["consensus_score"] for d in self.decision_history])
        )
        avg_outcome_confidence = float(
            np.mean([d.outcome_confidence for d in self.decision_history])
        )

        total_reviews = (
            self.trust_metrics.agreement_count + self.trust_metrics.disagreement_count
        )
        agreement_rate = (
            self.trust_metrics.agreement_count / total_reviews if total_reviews else 0.0
        )
        common_concerns = defaultdict(int)
        for decision in self.decision_history:
            for concern in decision.human_input["concerns"]:
                common_concerns[concern] += 1

        return {
            "authority_distribution": dict(authority_counts),
            "confidence_metrics": {
                "avg_ai_confidence": avg_ai_confidence,
                "avg_human_consensus": avg_human_consensus,
                "avg_outcome_confidence": avg_outcome_confidence,
            },
            "trust_evolution": {
                "current_human_trust": self.trust_metrics.human_trust_in_ai,
                "agreement_rate": agreement_rate,
                "total_decisions": len(self.decision_history),
            },
            "top_concerns": sorted(
                common_concerns.items(), key=lambda item: item[1], reverse=True
            )[:5],
            "partnership_maturity": self._calculate_partnership_maturity(),
        }

    def _calculate_partnership_maturity(self) -> str:
        """Assign a descriptive stage using decision count and proxy trust."""
        trust_level = self.trust_metrics.human_trust_in_ai
        decision_count = len(self.decision_history)

        if decision_count < 10:
            return "Forming"
        if trust_level < 0.6:
            return "Developing"
        if trust_level < 0.8:
            return "Optimizing"
        return "Mature"

## 4. Define the expert team and vessel scenario

The profiles and scenario below are illustrative inputs. Initial bay weights are sampled uniformly, so the starting state and the resulting rankings can differ between fresh runs. The senior-expert intuition rule also uses randomness. No empirical conclusions should be drawn from one simulated run.

In [ ]:
experts = [
    HumanExpert(
        id="EXP001",
        name="Captain Maria Rodriguez",
        experience_years=25,
        specialization="stability",
        confidence_level=0.92,
        vessel_knowledge={"MSC_Gulson": 0.95, "General": 0.88},
        risk_tolerance=0.3,
        decision_speed=0.8,
    ),
    HumanExpert(
        id="EXP002",
        name="Dr. James Chen",
        experience_years=18,
        specialization="hazardous",
        confidence_level=0.89,
        vessel_knowledge={"MSC_Gulson": 0.82, "General": 0.91},
        risk_tolerance=0.2,
        decision_speed=0.6,
    ),
    HumanExpert(
        id="EXP003",
        name="Sarah Thompson",
        experience_years=15,
        specialization="oversized",
        confidence_level=0.85,
        vessel_knowledge={"MSC_Gulson": 0.78, "General": 0.85},
        risk_tolerance=0.4,
        decision_speed=0.9,
    ),
    HumanExpert(
        id="EXP004",
        name="Michael Kumar",
        experience_years=22,
        specialization="operations",
        confidence_level=0.90,
        vessel_knowledge={"MSC_Gulson": 0.88, "General": 0.92},
        risk_tolerance=0.6,
        decision_speed=1.2,
    ),
    HumanExpert(
        id="EXP005",
        name="Lisa Anderson",
        experience_years=12,
        specialization="operations",
        confidence_level=0.78,
        vessel_knowledge={"MSC_Gulson": 0.72, "General": 0.80},
        risk_tolerance=0.5,
        decision_speed=1.0,
    ),
]

partnership = HumanAISymbioticPartnership(experts)
print(f"Expert team size: {len(experts)}")
print(f"Initial human trust proxy: {partnership.trust_metrics.human_trust_in_ai:.2f}")
print("Authority thresholds:")
for threshold, value in partnership.authority_allocation.items():
    print(f"  {threshold.replace('_', ' ').title()}: {value:.2f}")

In [ ]:
vessel_state = {
    "discharge_ports": ["Rotterdam", "Antwerp", "Singapore", "Dubai"],
    "weight_by_bay": defaultdict(float),
    "total_weight": 0.0,
    "current_loading_bay": 12,
}

for bay in range(1, 23):
    initial_bay_weight = float(np.random.uniform(50, 150))
    vessel_state["weight_by_bay"][bay] = initial_bay_weight
    vessel_state["total_weight"] += initial_bay_weight

test_containers = [
    {"id": "HAZ_001", "type": "hazardous", "weight": 15, "size": "40ft", "destination": "Singapore"},
    {"id": "OVZ_002", "type": "oversized", "weight": 35, "size": "45ft", "destination": "Rotterdam"},
    {"id": "STD_003", "type": "standard", "weight": 22, "size": "40ft", "destination": "Dubai"},
    {"id": "RF_004", "type": "reefer", "weight": 18, "size": "40ft", "destination": "Antwerp"},
    {"id": "HVY_005", "type": "standard", "weight": 28, "size": "40ft", "destination": "Singapore"},
]

available_slots = [
    "140284", "140286", "140208",
    "160284", "160286", "160208",
    "180284", "180286", "180208",
    "200284", "200286", "200208",
    "220284", "220286", "220208",
]

print(f"Initial vessel weight across modeled bays: {vessel_state['total_weight']:.1f} tons")
print(f"Discharge sequence: {' -> '.join(vessel_state['discharge_ports'])}")
print(f"Current loading bay: {vessel_state['current_loading_bay']}")
print(f"Containers in this scenario: {len(test_containers)}")
print(f"Candidate positions: {len(available_slots)}")

## 5. Run the collaborative decision scenarios

Each iteration ranks the positions still available, obtains expert review, records an authority decision, and then updates the modeled vessel state using the AI-suggested slot. Updating the bay weight and removing the selected position makes later recommendations depend on earlier placements. Since the workflow does not accept a human-selected replacement slot, this state update represents implementation of the AI suggestion for the purpose of the simulation, not a claim that human-led decisions always adopt it in practice.

In [ ]:
decisions_made = []

for index, container in enumerate(test_containers, start=1):
    if not available_slots:
        raise RuntimeError("No candidate positions remain for the next container.")

    decision = partnership.make_collaborative_decision(
        container, vessel_state, available_slots
    )
    decisions_made.append(decision)

    print(f"\nScenario {index}: {container['id']} ({container['type']})")
    print(f"  Context: {decision.problem_context}")
    print(f"  AI-suggested slot: {decision.ai_recommendation.suggested_slot}")
    print(f"  AI heuristic confidence: {decision.ai_recommendation.confidence_score:.3f}")
    print(f"  Illustrative benefit estimate: ${decision.ai_recommendation.expected_benefit:.2f}")
    print(f"  Expert consensus proxy: {decision.human_input['consensus_score']:.3f}")
    print(f"  Expert review label: {decision.human_input['recommendation']}")
    print(f"  Authority: {decision.decision_authority} ({decision.final_decision})")
    print(f"  Outcome-confidence proxy: {decision.outcome_confidence:.3f}")

    if decision.human_input["concerns"]:
        print("  Concerns: " + "; ".join(decision.human_input["concerns"]))
    if decision.human_input["suggestions"]:
        print("  Suggestions: " + "; ".join(decision.human_input["suggestions"]))

    chosen_slot = decision.ai_recommendation.suggested_slot
    bay = partnership.ai_engine._parse_bay(chosen_slot)
    vessel_state["weight_by_bay"][bay] += container["weight"]
    vessel_state["total_weight"] += container["weight"]
    vessel_state["current_loading_bay"] = bay
    available_slots.remove(chosen_slot)

    print("  Main scoring evidence:")
    for reason in decision.ai_recommendation.reasoning[:3]:
        print(f"    - {reason}")

print(f"\nRecorded decisions: {len(decisions_made)}")

## 6. Summarize the simulated partnership

The summary reports what the rule-based simulation recorded: authority counts, mean confidence indicators, a concern-based agreement rate, and recurring expert concerns. The agreement rate has a narrow operational definition here: a review counts as agreement when the selected experts recorded no concerns. It is not a direct measure of correctness, safety, or real-world human–AI agreement.

In [ ]:
performance = partnership.analyze_partnership_performance()

print("Partnership performance summary")
print("Authority distribution:")
for authority, count in sorted(performance["authority_distribution"].items()):
    share = count / len(decisions_made)
    print(f"  {authority}: {count} decisions ({share:.1%})")

confidence_metrics = performance["confidence_metrics"]
print("Confidence indicators (heuristic means):")
print(f"  AI confidence: {confidence_metrics['avg_ai_confidence']:.3f}")
print(f"  Human consensus: {confidence_metrics['avg_human_consensus']:.3f}")
print(f"  Combined outcome confidence: {confidence_metrics['avg_outcome_confidence']:.3f}")

trust_evolution = performance["trust_evolution"]
print("Trust-related indicators:")
print(f"  Current human-trust proxy: {trust_evolution['current_human_trust']:.3f}")
print(f"  Concern-based agreement rate: {trust_evolution['agreement_rate']:.1%}")
print(f"  Decision count: {trust_evolution['total_decisions']}")
print(f"  Descriptive maturity stage: {performance['partnership_maturity']}")

if performance["top_concerns"]:
    print("Most frequent concerns:")
    for concern, count in performance["top_concerns"]:
        print(f"  {concern}: {count} occurrence(s)")

## 7. Visualize decision and trust indicators

The dashboard uses the recorded decisions to display authority allocation, trust-proxy history, confidence indicators, expert review agreement rates, problem categories, and the descriptive maturity stage. Each plot is diagnostic: it visualizes the simulation's own rules and data rather than validating the model against real operational outcomes.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 15))
fig.suptitle("Human–AI Partnership: Simulated Decision Indicators", fontsize=16)

# Authority proportions show how often each rule-based authority category was assigned.
ax = axes[0, 0]
authority_data = performance["authority_distribution"]
ax.pie(
    list(authority_data.values()),
    labels=list(authority_data.keys()),
    autopct="%1.1f%%",
    startangle=90,
)
ax.set_title("Decision authority")

# Trust history stores the proxy value after each recorded decision.
ax = axes[0, 1]
trust_history = partnership.trust_metrics.collaboration_history
decision_numbers = range(1, len(trust_history) + 1)
trust_levels = [entry["human_trust_in_ai"] for entry in trust_history]
ax.plot(decision_numbers, trust_levels, marker="o", label="Human-trust proxy")
ax.axhline(0.5, color="gray", linestyle="--", label="Initial value")
ax.set_xlabel("Decision number")
ax.set_ylabel("Trust proxy (unitless)")
ax.set_title("Trust-proxy history")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend()

# Comparing these scores does not imply that they are calibrated probabilities.
ax = axes[1, 0]
decision_numbers = range(1, len(decisions_made) + 1)
ax.plot(
    decision_numbers,
    [d.ai_recommendation.confidence_score for d in decisions_made],
    marker="s",
    label="AI confidence heuristic",
)
ax.plot(
    decision_numbers,
    [d.human_input["consensus_score"] for d in decisions_made],
    marker="^",
    label="Human consensus proxy",
)
ax.plot(
    decision_numbers,
    [d.outcome_confidence for d in decisions_made],
    marker="D",
    label="Combined confidence proxy",
)
ax.set_xlabel("Decision number")
ax.set_ylabel("Heuristic score (unitless)")
ax.set_title("Confidence indicators")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend()

# Expert agreement is the share of that expert's recorded opinions above the chosen threshold.
ax = axes[1, 1]
expert_performance = defaultdict(lambda: {"agreements": 0, "total": 0})
for decision in decisions_made:
    for opinion in decision.human_input["expert_opinions"]:
        record = expert_performance[opinion["expert_name"]]
        record["total"] += 1
        if opinion["agreement_score"] > 0.7:
            record["agreements"] += 1

expert_names = list(expert_performance)
agreement_rates = [
    expert_performance[name]["agreements"] / expert_performance[name]["total"]
    for name in expert_names
]
ax.bar(expert_names, agreement_rates, color="steelblue", alpha=0.8)
ax.set_ylabel("Share of opinions above 0.7")
ax.set_title("Expert agreement-score rates")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", labelrotation=35)
ax.grid(axis="y", alpha=0.3)

# The category is recovered from the consistently formatted decision context.
ax = axes[2, 0]
problem_type_counts = defaultdict(int)
for decision in decisions_made:
    container_type = decision.problem_context.split("(", 1)[1].split(")", 1)[0]
    problem_type_counts[container_type] += 1
ax.bar(list(problem_type_counts), list(problem_type_counts.values()), color="darkcyan")
ax.set_xlabel("Container type")
ax.set_ylabel("Number of decisions")
ax.set_title("Decisions by container type")
ax.grid(axis="y", alpha=0.3)

# Maturity is a descriptive category, not a validated performance measure.
ax = axes[2, 1]
maturity_order = ["Forming", "Developing", "Optimizing", "Mature"]
maturity_scores = {name: (index + 1) / len(maturity_order) for index, name in enumerate(maturity_order)}
maturity = performance["partnership_maturity"]
ax.barh([maturity], [maturity_scores[maturity]], color="darkorange")
ax.set_xlim(0, 1)
ax.set_xlabel("Ordinal display scale (unitless)")
ax.set_title("Descriptive partnership maturity")
ax.grid(axis="x", alpha=0.3)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 8. Inspect a recommendation and a counterfactual

A counterfactual here compares the selected slot's heuristic score with the score stored for one alternative during the same recommendation. This is a score comparison, not a change in confidence or a guaranteed change in operational benefit. The risk indicators and expert opinions are shown alongside it so that the recommendation can be assessed as a decision-support artifact.

In [ ]:
sample_decision = decisions_made[1]
recommendation = sample_decision.ai_recommendation

print(f"Recommendation review: {sample_decision.problem_context}")
print(f"AI-suggested slot: {recommendation.suggested_slot}")
print(f"Heuristic score: {recommendation.recommendation_score:.3f}")
print(f"AI confidence heuristic: {recommendation.confidence_score:.3f}")
print(f"Illustrative benefit estimate: ${recommendation.expected_benefit:.2f}")

print("\nReasoning:")
for number, reason in enumerate(recommendation.reasoning, start=1):
    print(f"  {number}. {reason}")

print("\nIllustrative risk indicators:")
for risk_name, value in recommendation.risk_assessment.items():
    print(f"  {risk_name.replace('_', ' ').title()}: {value:.3f}")

if recommendation.alternatives:
    alternative = recommendation.alternatives[0]
    score_difference = recommendation.recommendation_score - alternative["score"]
    print("\nCounterfactual score comparison:")
    print(f"  Alternative slot: {alternative['slot']}")
    print(f"  Alternative heuristic score: {alternative['score']:.3f}")
    print(f"  Selected-slot score advantage: {score_difference:.3f}")
else:
    print("\nNo alternative slot was available for comparison.")

print("\nExpert opinions:")
for opinion in sample_decision.human_input["expert_opinions"]:
    print(
        f"  {opinion['expert_name']} ({opinion['specialization']}): "
        f"agreement proxy {opinion['agreement_score']:.3f}"
    )
    print(f"    Review basis: {opinion['reasoning']}")
    if opinion["concerns"]:
        print("    Concerns: " + "; ".join(opinion["concerns"]))
    if opinion["suggestions"]:
        print("    Suggestions: " + "; ".join(opinion["suggestions"]))

## 9. Interpreting the model

The notebook demonstrates how a decision-support workflow can make its stages visible: a weighted slot score leads to a ranked recommendation, a specialization-based review adds concerns and suggestions, and a threshold rule assigns authority. The record also makes it possible to inspect how the simulation's own trust proxy changes over time.

These mechanisms do not establish that collaboration is more accurate than either human-only or AI-only planning. The score weights, risk values, benefit calculation, confidence thresholds, and trust updates are illustrative rules. The initial bay weights and some expert opinions are stochastic, and the small scenario is not a representative sample. Any operational use would require validated vessel and cargo data, explicit hard constraints, safety review, and evaluation against observed outcomes.

In [ ]:
final_performance = partnership.analyze_partnership_performance()
final_metrics = final_performance["confidence_metrics"]
final_trust = final_performance["trust_evolution"]

print("Human–AI partnership simulation summary")
print(f"  Decisions recorded: {final_trust['total_decisions']}")
print(f"  Descriptive maturity stage: {final_performance['partnership_maturity']}")
print(f"  Human-trust proxy: {final_trust['current_human_trust']:.3f}")
print(f"  Concern-based agreement rate: {final_trust['agreement_rate']:.1%}")
print(f"  Mean AI confidence heuristic: {final_metrics['avg_ai_confidence']:.3f}")
print(f"  Mean human consensus proxy: {final_metrics['avg_human_consensus']:.3f}")
print(f"  Mean combined confidence proxy: {final_metrics['avg_outcome_confidence']:.3f}")
print("\nInterpretation: these are outputs of the stated simulation rules, not validated operational performance measures.")

## Conclusion

This example combines explainable heuristic slot ranking, specialization-based expert review, threshold-based authority allocation, and trust-indicator tracking. Its main educational value is the explicit connection between inputs, scoring rules, human-review logic, recorded decisions, and summary diagnostics.

The recommendations remain advisory. A high score or confidence indicator does not certify physical feasibility, stability, safety, or compliance. The rule-based trust and maturity measures describe only this simulation, and the illustrative benefit estimate should not be treated as realized savings. Further work within this modeling approach could add vessel-specific data, explicit operational constraints, and outcome-based validation while preserving the human-review and authority-allocation workflow.